# Workshop 8 — Text Preprocessing Pipeline
**Saharsh Pathak | 2417371 | Herald College Kathmandu**

Building a complete NLP preprocessing pipeline: tokenisation, stopword removal,
stemming, lemmatisation, TF-IDF, and word embeddings.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import string
from collections import Counter

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

print('NLP libraries loaded')

## 1. Sample Corpus

In [ ]:
corpus = [
    "Artificial intelligence is transforming the way we live and work.",
    "Machine learning algorithms learn patterns from large datasets.",
    "Deep learning uses neural networks with many hidden layers.",
    "Natural language processing enables computers to understand human text.",
    "Computer vision allows machines to interpret and understand images.",
    "Reinforcement learning trains agents through rewards and penalties.",
    "Transfer learning leverages pre-trained models for new tasks.",
    "Data preprocessing is a critical step in any machine learning pipeline."
]

print(f'Corpus size: {len(corpus)} documents')
for i, doc in enumerate(corpus):
    print(f'  Doc {i+1}: {doc[:60]}...')

## 2. Tokenisation

In [ ]:
# Word tokenisation
sample = corpus[0]
word_tokens = word_tokenize(sample)
sent_tokens = sent_tokenize(' '.join(corpus[:3]))

print('Original:', sample)
print('\nWord tokens:', word_tokens)
print(f'Token count: {len(word_tokens)}')
print('\nSentence tokens:', sent_tokens)

## 3. Text Cleaning

In [ ]:
def clean_text(text):
    """Full text cleaning pipeline."""
    text = text.lower()                                    # 1. Lowercase
    text = re.sub(r'http\S+|www\S+', '', text)            # 2. Remove URLs
    text = re.sub(r'<.*?>', '', text)                      # 3. Remove HTML tags
    text = re.sub(r'[^a-z\s]', '', text)                  # 4. Remove punctuation/numbers
    text = re.sub(r'\s+', ' ', text).strip()              # 5. Remove extra whitespace
    return text

cleaned = [clean_text(doc) for doc in corpus]
print('Before:', corpus[0])
print('After: ', cleaned[0])

## 4. Stopword Removal

In [ ]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    tokens = word_tokenize(text)
    return ' '.join([w for w in tokens if w not in stop_words])

no_stop = [remove_stopwords(doc) for doc in cleaned]
print('With stopwords:    ', cleaned[0])
print('Without stopwords: ', no_stop[0])
print(f'\nTokens removed: {len(cleaned[0].split()) - len(no_stop[0].split())}')

## 5. Stemming vs Lemmatisation

In [ ]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

test_words = ['running', 'learning', 'algorithms', 'transforming', 'networks', 'layers', 'enables']

print(f'{'Word':20s} {'Stemmed':20s} {'Lemmatised':20s}')
print('-' * 60)
for word in test_words:
    stemmed = stemmer.stem(word)
    lemmatized = lemmatizer.lemmatize(word, pos='v')
    print(f'{word:20s} {stemmed:20s} {lemmatized:20s}')

## 6. TF-IDF Vectorisation

In [ ]:
tfidf = TfidfVectorizer(max_features=20, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(no_stop)

df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=tfidf.get_feature_names_out()
)
print('TF-IDF Matrix shape:', tfidf_matrix.shape)
print(df_tfidf.round(3))

# Visualise TF-IDF heatmap
plt.figure(figsize=(14, 5))
plt.imshow(df_tfidf.values, aspect='auto', cmap='YlOrRd')
plt.colorbar(label='TF-IDF Score')
plt.xticks(range(len(df_tfidf.columns)), df_tfidf.columns, rotation=45, ha='right', fontsize=8)
plt.yticks(range(len(corpus)), [f'Doc {i+1}' for i in range(len(corpus))])
plt.title('TF-IDF Matrix Heatmap')
plt.tight_layout(); plt.show()

## 7. Word Frequency Analysis

In [ ]:
all_words = ' '.join(no_stop).split()
word_freq = Counter(all_words)
top_words = word_freq.most_common(15)

words, freqs = zip(*top_words)
plt.figure(figsize=(10, 5))
plt.bar(words, freqs, color='#0EA5E9')
plt.title('Top 15 Most Frequent Words (after stopword removal)')
plt.xlabel('Word'); plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()